# Wessex Stage 3A CPU replay / resume

This notebook replays only the saved Stage 3A development diagnostics. It does not launch Stage 3B, final-period evaluation, intervals or a software application. SiteID identities and historical QA availability remain unverified. Use a separate Drive folder so the earlier project is preserved. The CPU experiment and scientific code were actually validated locally; a live Colab/Drive session is not claimed.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, zipfile, shutil, stat, subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DEFAULT_ROOT = '/content/drive/MyDrive/Wastewater_Stage3A'
except ImportError:
    DEFAULT_ROOT = '/workspace/Wastewater_Paper'
PROJECT_ROOT = Path(DEFAULT_ROOT)
ZIP_DIRECTORY = Path('/content/drive/MyDrive/Wastewater_Deliveries')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)


Place all new Stage 3A ZIP parts and the three unchanged Stage 2 archives in ZIP_DIRECTORY. Also put the new delivery SHA256SUMS there. Each ZIP must remain below 24 MiB. Stage 2 Code is retained as an original archive; its old document payload is not overlaid on the Stage 3A package. Runs and Reports are imported into their original versioned paths.


In [ ]:
OLD_SHA = {
 'Wastewater_Stage2_Code_Data_20261010.zip':'9e1563811e35b0d9abafaf59e9d11342d4b507520f8009181b803ff71ec7c5ed',
 'Wastewater_Stage2_Runs_20261010.zip':'2d234a71243574acd7ebd22720c340f299ead73a2cc579e4fb895cd8c71962c7',
 'Wastewater_Stage2_Reports_Figures_20261010.zip':'fb833734ed1c34fcbf819fb997a3d2834dd08caa9ced61f6bc76494f5b203be3'}
def digest_bytes(b): return hashlib.sha256(b).hexdigest()
def safe_import_zip(archive, root, expected_sha=None, extract_payload=True):
    archive, root = Path(archive), Path(root).resolve()
    assert archive.stat().st_size < 24*1024**2, 'ZIP exceeds limit'
    if expected_sha is not None:
        assert digest_bytes(archive.read_bytes()) == expected_sha, 'Archive SHA256 mismatch'
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None, 'CRC failure'
        names = z.namelist(); assert len(names) == len(set(names)), 'Duplicate ZIP member'
        assert 'SHA256SUMS' in names, 'Missing checksum manifest'
        hashes = {}
        for line in z.read('SHA256SUMS').decode().splitlines():
            checksum, name = line.split('  ', 1); hashes[name] = checksum
        assert set(hashes) == set(names)-{'SHA256SUMS'}, 'Member manifest mismatch'
        for name, checksum in hashes.items():
            p = Path(name)
            assert not p.is_absolute() and '..' not in p.parts, 'Unsafe member path'
            assert not stat.S_ISLNK(z.getinfo(name).external_attr >> 16), 'ZIP symlink'
            target = (root/p).resolve(); assert target.is_relative_to(root), 'Escaping path'
            payload = z.read(name); assert digest_bytes(payload) == checksum, 'Member SHA mismatch'
            if extract_payload:
                if target.exists():
                    assert target.is_file() and digest_bytes(target.read_bytes()) == checksum, 'Conflicting existing file; use a new ROOT'
                else:
                    target.parent.mkdir(parents=True, exist_ok=True); target.write_bytes(payload)
    return len(hashes)

new_sha = {}
for line in (ZIP_DIRECTORY/'SHA256SUMS').read_text().splitlines():
    h, name = line.split('  ', 1); new_sha[name] = h
new_archives = sorted(ZIP_DIRECTORY.glob('Wastewater_Stage3A_*.zip'))
assert new_archives, 'No Stage 3A ZIPs'
for p in new_archives:
    assert p.name in new_sha, 'Archive absent from delivery manifest'
    print(p.name, safe_import_zip(p, PROJECT_ROOT, new_sha[p.name]))
for name, h in OLD_SHA.items():
    p=ZIP_DIRECTORY/name; assert p.is_file(), 'Missing original Stage 2 archive'
    safe_import_zip(p, PROJECT_ROOT, h, extract_payload=('Code_Data' not in name))
    for folder in ['deliverables/stage2_20261010','preservation/stage2_before_stage3a_20261010']:
        dest=PROJECT_ROOT/folder/name; dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists(): assert digest_bytes(dest.read_bytes())==h
        else: shutil.copy2(p,dest)
print('Archives and member checksums verified; source roots preserved.')


In [ ]:
subprocess.run(['bash','scripts/refresh_stage2_environment.sh'],cwd=PROJECT_ROOT,check=True)
PYTHON = PROJECT_ROOT/'.venv-stage2/bin/python'
RUN_ENV = os.environ.copy()
RUN_ENV.update(PYTHONPATH=str(PROJECT_ROOT/'src'), OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1', MPLCONFIGDIR=str(PROJECT_ROOT/'.cache/matplotlib'))
cfg=json.loads((PROJECT_ROOT/'configs/stage3a_20261010/full.json').read_text())
assert cfg['version']=='stage3a_20261010_v3' and cfg['seeds']==[20261009,20261010,20261011]
assert cfg['source_training_end']=='2021-12-31' and cfg['budgets']==[30,90,180] and cfg['horizons']==[1,3]
subprocess.run([str(PYTHON),'-m','unittest','discover','-s','tests','-p','test_stage3a.py','-v'],cwd=PROJECT_ROOT,env=RUN_ENV,check=True)


Execution defaults to verifying saved completed caches. On a fresh compute run, trajectories save optimizer and shuffle-generator state every 20 epochs and at candidate endpoints. Interrupted conditions retain checkpoints and failure logs; rerun the same immutable configuration to resume. Changed scientific code/config/data must use a new version. Drive paths persist across notebook sessions. GPU is not used or required; do not switch device silently.


In [ ]:
def run_logged(arguments, name):
    folder=PROJECT_ROOT/'validation'; folder.mkdir(exist_ok=True)
    log=folder/name
    with log.open('a') as f:
        p=subprocess.run([str(PYTHON),*arguments],cwd=PROJECT_ROOT,env=RUN_ENV,stdout=f,stderr=subprocess.STDOUT)
    print(log, 'exit=', p.returncode)
    if p.returncode: raise RuntimeError('Failed; preserved log/checkpoints. Inspect before resume.')
run_logged(['-m','wastewater.stage3a_experiment','--smoke','--workers','2'],'stage3a_colab_smoke.log')
run_logged(['-m','wastewater.stage3a_experiment','--workers','2'],'stage3a_colab_full.log')
manifest=json.loads((PROJECT_ROOT/cfg['output']/'manifest.json').read_text())
assert manifest['status']=='completed' and manifest['completed_conditions']==240 and not manifest['failed_conditions']
print('Stage 3A completed; STOP before Stage 3B.')


The delivery already contains checked tables and vector figures. Reporting scripts can reproduce them from completed predictions without retraining. The local 41-test suite, two-site smoke and full CPU experiment provide actual validation evidence; real Drive mounts, notebook runtime and GPU behavior have not been verified.
